# MyCON — اجرای یکپارچهٔ پایلوت
ویدیوی پروژه را انتخاب کنید و سلول‌ها را به‌ترتیب یا با Run all اجرا کنید. اجرای CPU، دوربین‌های COLMAP، هندسهٔ پراکندهٔ غیرمتریک، Shape 3 و Viewer آفلاین را تولید می‌کند. استریوی متراکم به CUDA نیاز دارد و آماده‌سازی دادهٔ Gaussian Splatting فقط از فریم‌های آموزش انجام می‌شود.

In [ ]:
from datetime import UTC, datetime
from pathlib import Path

VIDEO = ""  # Absolute path to a video in Colab or mounted Drive
TEST_VIDEO = ""  # Optional distinct capture of the same scene
CAPTURE_ID = "site01-session01"
TEST_CAPTURE_ID = "site01-heldout"
EXPECTED_VIDEO_SHA256 = ""  # Optional hash recorded before processing
RUNS_DIR = "/content/mycon-runs"
CODE_REF = "59788186d5926c967d58d9a40f69878c0717f5d0"  # Frozen implementation revision
LONG_VIDEO = True  # Full-capture disk-backed scan; use False for old short pilot
SAMPLE_HZ = 8.0
MAX_FRAMES = 1200  # per source; increase for longer captures after storage check
ATTEMPT_DENSE = False
GEOMETRY_THRESHOLD_UNSCALED = 0.075
REVIEW_ID = "pilot-" + datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
run = Path(RUNS_DIR) / ("recon-" + REVIEW_ID)
if not VIDEO:
    raise ValueError("Set VIDEO to the absolute path of an input video in this cell.")
if VIDEO.startswith("/content/drive/") or TEST_VIDEO.startswith("/content/drive/") or RUNS_DIR.startswith("/content/drive/"):
    from google.colab import drive
    drive.mount("/content/drive")
if not Path(VIDEO).is_file():
    raise FileNotFoundError(VIDEO)

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile

checkout = Path(tempfile.mkdtemp(prefix="mycon-code-"))
subprocess.run(["git","init","-q",str(checkout)],check=True)
subprocess.run(["git","-C",str(checkout),"fetch","--depth","1",
                "https://github.com/sayyedalimrj/MyCon.git",CODE_REF],check=True)
subprocess.run(["git","-C",str(checkout),"checkout","--detach","FETCH_HEAD"],check=True)
revision = subprocess.check_output(["git","-C",str(checkout),"rev-parse","HEAD"],text=True).strip()
repo = checkout / "repository"
if not shutil.which("colmap"):
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","colmap"],check=True)
venv = checkout / ".venv"
subprocess.run([sys.executable,"-m","venv",str(venv)],check=True)
pilot_python = venv / "bin/python"
subprocess.run([str(pilot_python),"-m","pip","install","-r",str(repo/"requirements-work-pilot.txt")],check=True)
if ATTEMPT_DENSE and shutil.which("nvidia-smi"):
    subprocess.run([str(pilot_python),"-m","pip","install","pycolmap-cuda12==3.13.0"],check=True)
print("MyCON commit", revision)

In [ ]:
import json

env = dict(os.environ, QT_QPA_PLATFORM="offscreen", OPENBLAS_NUM_THREADS="1")
command = [str(pilot_python),"-m","reconstruction.consolidated_pilot","--run-dir",str(run),
           "--review-id",REVIEW_ID,"--video",VIDEO,"--capture-id",CAPTURE_ID,
           "--geometry-threshold",str(GEOMETRY_THRESHOLD_UNSCALED)]
if TEST_VIDEO:
    command.extend(["--test-video",TEST_VIDEO,"--test-capture-id",TEST_CAPTURE_ID])
if EXPECTED_VIDEO_SHA256:
    command.extend(["--expected-video-sha256",EXPECTED_VIDEO_SHA256])
if LONG_VIDEO:
    command.extend(["--long-video", "--sample-hz", str(SAMPLE_HZ), "--max-frames", str(MAX_FRAMES)])
if ATTEMPT_DENSE:
    command.append("--attempt-dense")
result = subprocess.run(command,cwd=repo,env=env)
receipt = run / f"pilot-{REVIEW_ID}.json"
if receipt.is_file():
    print(receipt)
    print(json.loads(receipt.read_text())["status"])
    with (run/"code_commit.txt").open("x") as stream:
        stream.write(revision+"\n")
result.check_returncode()

## خواندن نتایج
پوشهٔ `offline-...` را به‌صورت یکجا باز یا دانلود کنید. Shape 4 و درصد پیشرفت به کنترل‌های مستقل، انطباق تأییدشده و مقادیر اجرایی نیاز دارند. دستورهای بررسی IFC، Stage 8 تا 11 و بازیابی در راهنمای شاخه ثبت شده‌اند. برای تکرار بررسی یک SfM کامل، شناسهٔ بررسی جدید بدهید و گزینهٔ ویدیو را حذف کنید.